# GTR + RADAR on Colab (Python 3.12, single A100, fp32 + TF32)

1. Runtime → Change runtime type → **A100 GPU**.
2. Push the repo (with `radar/`, `exp/exp_radar.py`, `run_radar.py`, `run_orig.py`, `scripts/RADAR/`, `tests/`) to GitHub, or upload it to Drive.
3. Put the original dataset files in a Drive folder (`ETTh1.csv`, `ETTm1.csv`, `weather.csv`, `ETTh2.csv`, `ETTm2.csv`, `solar_AL.txt`, `electricity.csv`, `traffic.csv`). Nothing is generated synthetically; a missing file stops the pipeline.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
import torch, numpy, pandas, sklearn
print('torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/knull-cc/GTR_RADAR.git'      # or copy the repo from Drive instead
DATA_SRC = '/content/drive/MyDrive/datasets'                 # folder holding the dataset files
SAVE_DIR = '/content/drive/MyDrive/GTR_RADAR_runs'           # results / figures / logs are copied here

In [ ]:
%cd /content
!test -d GTR_RADAR || git clone $REPO_URL GTR_RADAR
%cd /content/GTR_RADAR
!git log --oneline -1
!mkdir -p dataset && cp -n "$DATA_SRC"/* dataset/ 2>/dev/null; ls -la dataset

## 1 · Unit tests (B=4, L=96, C=7, H=96, cycle=24; random GTR in eval mode)

In [ ]:
!pip -q install pytest
!python -m pytest -q tests/test_radar.py

## 2 · Quick pipeline on ETTh1 (seed 2024, H=96)
original `run.py` (reproduction check, must be within 2%) → S0 → S1 → S2 → S3 joint → S3 plug-in → eval → summarize → plot.
Checkpoints are reused when they exist (`SKIP_EXISTING=1`); set `SKIP_EXISTING=0` to retrain.

In [ ]:
!bash scripts/RADAR/run_quick.sh ETTh1

In [ ]:
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open('results_radar/summary.md').read()))
for f in sorted(glob.glob('figures_radar/*.png')):
    print(f); display(Image(f, width=900))

## 3 · Quick runs on ETTm1 and Weather

In [ ]:
!bash scripts/RADAR/run_quick.sh ETTm1
!bash scripts/RADAR/run_quick.sh weather
!python run_radar.py summarize

In [ ]:
!mkdir -p "$SAVE_DIR" && cp -r results_radar figures_radar logs_radar "$SAVE_DIR"/ && ls "$SAVE_DIR"

## 4 · Full grid (pred_len 96/192/336/720 × seeds 2024/2025/2026), priority order
ETTh1, ETTm1, Weather → ETTh2, ETTm2, Solar → Electricity, Traffic. Long-running; checkpoints live in `checkpoints_radar/`
(point `CKPT_DIR` at Drive if the session may disconnect).

In [ ]:
!CKPT_DIR=$SAVE_DIR/checkpoints_radar/ bash scripts/RADAR/run_all.sh etth1 ettm1 weather

In [ ]:
!CKPT_DIR=$SAVE_DIR/checkpoints_radar/ bash scripts/RADAR/run_all.sh etth2 ettm2 solar electricity traffic
!mkdir -p "$SAVE_DIR" && cp -r results_radar figures_radar logs_radar "$SAVE_DIR"/